In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# 1. Generate a More Challenging High-Dimensional Dataset
n_samples = 200
n_informative = 5  # Very few truly important features
n_redundant = 50   # A large number of redundant features
n_noisy = 45       # Significant amount of random noise features
n_features = n_informative + n_redundant + n_noisy
random_state = 42

X, y = make_classification(n_samples=n_samples,
                           n_features=n_features,
                           n_informative=n_informative,
                           n_redundant=n_redundant,
                           n_repeated=0,  # Ensure no perfectly repeated features
                           n_classes=2,
                           flip_y=0.05,  # Introduce a small amount of label noise
                           random_state=random_state)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=random_state)

print(f"Original training data shape: {X_train.shape}")
print(f"Original testing data shape: {X_test.shape}")

# 2. Train a Logistic Regression model on the original high-dimensional data (with scaling)
pipeline_high_dim = Pipeline([
    ('scaler', StandardScaler()),
    ('logistic', LogisticRegression(solver='liblinear', random_state=random_state))
])
pipeline_high_dim.fit(X_train, y_train)
y_pred_high_dim = pipeline_high_dim.predict(X_test)
accuracy_high_dim = accuracy_score(y_test, y_pred_high_dim)
print(f"Accuracy on high-dimensional data: {accuracy_high_dim:.2f}")

# 3. Apply PCA for dimensionality reduction within a pipeline
n_components = 5  # Reduce to the number of informative features
pipeline_low_dim = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=n_components)),
    ('logistic', LogisticRegression(solver='liblinear', random_state=random_state))
])
pipeline_low_dim.fit(X_train, y_train)
y_pred_low_dim = pipeline_low_dim.predict(X_test)
accuracy_low_dim = accuracy_score(y_test, y_pred_low_dim)
print(f"Accuracy on reduced data: {accuracy_low_dim:.2f}")
print(f"Explained variance ratio of the first {n_components} components: {np.sum(pipeline_low_dim.named_steps['pca'].explained_variance_ratio_):.2f}")

# 4. Visualize the effect (if reduced to 2 or 3 components)
if n_components == 2:
    X_test_reduced_viz = pipeline_low_dim.named_steps['pca'].transform(pipeline_low_dim.named_steps['scaler'].transform(X_test))
    plt.figure(figsize=(8, 6))
    plt.scatter(X_test_reduced_viz[:, 0], X_test_reduced_viz[:, 1], c=y_test, cmap='viridis')
    plt.title('Reduced Test Data (2 Principal Components)')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.colorbar(label='Class')
    plt.tight_layout()
    plt.show()
elif n_components == 3:
    X_test_reduced_viz = pipeline_low_dim.named_steps['pca'].transform(pipeline_low_dim.named_steps['scaler'].transform(X_test))
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    scatter = ax.scatter(X_test_reduced_viz[:, 0], X_test_reduced_viz[:, 1], X_test_reduced_viz[:, 2], c=y_test, cmap='viridis')
    ax.set_title('Reduced Test Data (3 Principal Components)')
    ax.set_xlabel('Principal Component 1')
    ax.set_ylabel('Principal Component 2')
    ax.set_zlabel('Principal Component 3')
    fig.colorbar(scatter, label='Class')
    plt.tight_layout()
    plt.show()
else:
    print("\nVisualization not shown as the reduced dimensionality is not 2 or 3.")
